In [1]:
import plotly.express as px
import pandas as pd
from extract import fetch_yield_curve_archive_xml
from transform import create_yield_curve_dataframe, parse_yield_curve_records
from missing_data import handle_missing_data

In [2]:
xml_bytes = fetch_yield_curve_archive_xml(1990, 2023)
df = create_yield_curve_dataframe(handle_missing_data(parse_yield_curve_records(xml_bytes)))

2026-11-04 21:23:27 [INFO] - 응답 완료: 1990-2023 (상태 코드: 200)
2026-11-04 21:23:27 [INFO] - 금리 레코드 파싱 완료: 8507건
2026-11-04 21:23:27 [INFO] - DataFrame 생성 완료: 12415행, 기간 1990-01-02 00:00:00 ~ 2023-12-29 00:00:00


In [ ]:
CURVE_LABELS = {
    "date": "Date",
    "yield_1m": "1M",
    "yield_1_5m": "1.5M",
    "yield_2m": "2M",
    "yield_3m": "3M",
    "yield_4m": "4M",
    "yield_6m": "6M",
    "yield_1y": "1Y",
    "yield_2y": "2Y",
    "yield_3y": "3Y",
    "yield_5y": "5Y",
    "yield_7y": "7Y",
    "yield_10y": "10Y",
    "yield_20y": "20Y",
    "yield_30y": "30Y",
}

chart_df = df.rename(columns=CURVE_LABELS)

figure = px.line(
    chart_df,
    x="Date",
    y=[chart_label for df_label, chart_label in CURVE_LABELS.items() if df_label.startswith("yield_")],
    title="US Treasury Yield Curve",
    labels={
        "Date": "Date",
        "value": "Yield (%)",
        "variable": "Maturity",
    },
)

figure.update_layout(
    template="plotly_white",
    hovermode="x unified",
    legend_title="Maturity",
)

VISIBLE_YIELDS = ["3M", "2Y", "10Y"]

figure.for_each_trace(
    lambda trace: trace.update(visible=True)
    if trace.name in VISIBLE_YIELDS
    else trace.update(visible="legendonly")
)

figure.update_xaxes(rangeslider_visible=True)

figure.show()

In [ ]:
from load_csv import _to_spread

SPREAD_LABLES = {
    "date": "Date",
    "spread_10y_2y": "T10Y2Y",
    "spread_10y_3m": "T10Y3M",
}
spread_df = _to_spread(df)

chart_df = spread_df.rename(columns=SPREAD_LABLES)
    
figure = px.line(
    chart_df,
    x="Date",
    y=[chart_label for df_label, chart_label in SPREAD_LABLES.items() if df_label.startswith("spread_")],
    title="US Treasury Yield Spread",
    labels={
        "Date": "Date",
        "value": "Spread(%p)",
        "variable": "Spread",
    },
)

figure.update_layout(
    template="plotly_white",
    hovermode="x unified",
    legend_title="Spread",
)
figure.add_hline(
    y=0,
    line_dash="dash",
    line_color="gray",
    line_width=1,
)

figure.update_xaxes(rangeslider_visible=True)

figure.show()
